**Find and Terminate EC2 Instances with SSH Open to the World**

**What this script does:**
1. Finds all security groups that allow SSH (port 22) from anywhere (0.0.0.0/0)
2. Finds all running or stopped EC2 instances using those security groups
3. Terminates those EC2 instances

**Why this matters:** Leaving SSH open to the world is a major security risk. This is the type of automation you'd build with AWS Lambda to keep your environment secure.

**Step 1: Import Modules and Connect to AWS**

In [34]:
import boto3
import json

ec2_client = boto3.client("ec2")

**Step 2: Get All Security Group Rules in the Region**

In [35]:
response = ec2_client.describe_security_group_rules()

**See what the response looks like**

**Get the value of the "SecurityGroupRules" key from the response**

In [36]:
all_sg_rules = response["SecurityGroupRules"]

**See what the security group rules look like**

In [37]:
print(json.dumps(all_sg_rules, indent=2, default=str))

[
  {
    "SecurityGroupRuleId": "sgr-0bf06981e91b6d089",
    "GroupId": "sg-0f9b6b99fcd41210d",
    "GroupOwnerId": "127486921697",
    "IsEgress": true,
    "IpProtocol": "-1",
    "FromPort": -1,
    "ToPort": -1,
    "CidrIpv4": "0.0.0.0/0",
    "Tags": [],
    "SecurityGroupRuleArn": "arn:aws:ec2:eu-west-2:127486921697:security-group-rule/sgr-0bf06981e91b6d089"
  },
  {
    "SecurityGroupRuleId": "sgr-0092c6e58c01caee6",
    "GroupId": "sg-0520e786f1be8713e",
    "GroupOwnerId": "127486921697",
    "IsEgress": false,
    "IpProtocol": "tcp",
    "FromPort": 22,
    "ToPort": 22,
    "CidrIpv4": "0.0.0.0/0",
    "Tags": [],
    "SecurityGroupRuleArn": "arn:aws:ec2:eu-west-2:127486921697:security-group-rule/sgr-0092c6e58c01caee6"
  },
  {
    "SecurityGroupRuleId": "sgr-0f623602249b9b3ec",
    "GroupId": "sg-0f9b6b99fcd41210d",
    "GroupOwnerId": "127486921697",
    "IsEgress": false,
    "IpProtocol": "tcp",
    "FromPort": 22,
    "ToPort": 22,
    "CidrIpv4": "78.146.96.193/32",

**Step 3: Find Security Groups Where SSH (Port 22) is Open to the World (0.0.0.0/0)**

In [38]:
open_ssh_sg_ids = []

for rule in all_sg_rules:
    if rule.get("FromPort") == 22 and rule.get("CidrIpv4") == "0.0.0.0/0":
        open_ssh_sg_ids.append(rule["GroupId"])


See which security group IDs have SSH open to the world

In [39]:
print(json.dumps(open_ssh_sg_ids, indent=2, default=str))

[
  "sg-0520e786f1be8713e",
  "sg-055b7a023bc18417c",
  "sg-084ce9c90d3d8510d"
]


**Step 4: Get All EC2 Instances in the Region**

In [40]:
response = ec2_client.describe_instances()

**See what the response looks like**

In [41]:
print(json.dumps(response, indent=2, default=str))

{
  "Reservations": [
    {
      "ReservationId": "r-0fed3dd41bab7a011",
      "OwnerId": "127486921697",
      "Groups": [],
      "Instances": [
        {
          "Architecture": "x86_64",
          "BlockDeviceMappings": [
            {
              "DeviceName": "/dev/xvda",
              "Ebs": {
                "AttachTime": "2026-05-29 12:47:36+00:00",
                "DeleteOnTermination": true,
                "Status": "attached",
                "VolumeId": "vol-0baa2c2c49830d5a3",
                "EbsCardIndex": 0
              }
            }
          ],
          "ClientToken": "5f79b31d-2b44-4d9e-87f4-c5092f6081f8",
          "EbsOptimized": false,
          "EnaSupport": true,
          "Hypervisor": "xen",
          "NetworkInterfaces": [
            {
              "Association": {
                "IpOwnerId": "amazon",
                "PublicDnsName": "ec2-35-178-9-218.eu-west-2.compute.amazonaws.com",
                "PublicIp": "35.178.9.218"
              },


**Get the value of the "Reservations" key from the response**

In [42]:
reservations = response["Reservations"]

**See what the reservations look like**

In [43]:
print(json.dumps(reservations, indent=2, default=str))

[
  {
    "ReservationId": "r-0fed3dd41bab7a011",
    "OwnerId": "127486921697",
    "Groups": [],
    "Instances": [
      {
        "Architecture": "x86_64",
        "BlockDeviceMappings": [
          {
            "DeviceName": "/dev/xvda",
            "Ebs": {
              "AttachTime": "2026-05-29 12:47:36+00:00",
              "DeleteOnTermination": true,
              "Status": "attached",
              "VolumeId": "vol-0baa2c2c49830d5a3",
              "EbsCardIndex": 0
            }
          }
        ],
        "ClientToken": "5f79b31d-2b44-4d9e-87f4-c5092f6081f8",
        "EbsOptimized": false,
        "EnaSupport": true,
        "Hypervisor": "xen",
        "NetworkInterfaces": [
          {
            "Association": {
              "IpOwnerId": "amazon",
              "PublicDnsName": "ec2-35-178-9-218.eu-west-2.compute.amazonaws.com",
              "PublicIp": "35.178.9.218"
            },
            "Attachment": {
              "AttachTime": "2026-05-29 12:47:36+00:

**Step 5: Filter Out Terminated Instances and Keep Only Running or Stopped Instances**

In [44]:
active_instances = []

for reservation in reservations:
    for instance in reservation["Instances"]:
        if instance["State"]["Name"] != "terminated":
            active_instances.append(instance)

**See the list of active instances**

In [45]:
print(json.dumps(active_instances, indent=2, default=str))

[
  {
    "Architecture": "x86_64",
    "BlockDeviceMappings": [
      {
        "DeviceName": "/dev/xvda",
        "Ebs": {
          "AttachTime": "2026-05-29 12:47:36+00:00",
          "DeleteOnTermination": true,
          "Status": "attached",
          "VolumeId": "vol-0baa2c2c49830d5a3",
          "EbsCardIndex": 0
        }
      }
    ],
    "ClientToken": "5f79b31d-2b44-4d9e-87f4-c5092f6081f8",
    "EbsOptimized": false,
    "EnaSupport": true,
    "Hypervisor": "xen",
    "NetworkInterfaces": [
      {
        "Association": {
          "IpOwnerId": "amazon",
          "PublicDnsName": "ec2-35-178-9-218.eu-west-2.compute.amazonaws.com",
          "PublicIp": "35.178.9.218"
        },
        "Attachment": {
          "AttachTime": "2026-05-29 12:47:36+00:00",
          "AttachmentId": "eni-attach-07b67d56256065ccc",
          "DeleteOnTermination": true,
          "DeviceIndex": 0,
          "Status": "attached",
          "NetworkCardIndex": 0
        },
        "Descriptio

**Step 6: Get Each Instance ID and Its Security Groups**

In [46]:
instance_info = []

for instance in active_instances:
    instance_id = instance["InstanceId"]
    sg_ids = [sg["GroupId"] for sg in instance["SecurityGroups"]]
    instance_info.append([instance_id, sg_ids])

**See each instance ID and its associated security groups**

In [47]:
print(json.dumps(instance_info, indent=2, default=str))

[
  [
    "i-018c484b8e50fe20c",
    [
      "sg-084ce9c90d3d8510d"
    ]
  ],
  [
    "i-0178ce093ceb0bedf",
    [
      "sg-0520e786f1be8713e"
    ]
  ],
  [
    "i-026c24bffde7c19f7",
    [
      "sg-06727ae78199b4e46"
    ]
  ]
]


**Step 7: Find Instances That Have a Security Group with SSH Open to the World**

In [48]:
instances_to_terminate = []

for item in instance_info:
    instance_id = item[0]
    sg_ids = item[1]

    for sg_id in sg_ids:
        if sg_id in open_ssh_sg_ids:
            instances_to_terminate.append(instance_id)

**See which instances will be terminated**

In [49]:
print(json.dumps(instances_to_terminate, indent=2, default=str))

[
  "i-018c484b8e50fe20c",
  "i-0178ce093ceb0bedf"
]


**Step 8: Terminate the Insecure Instances**

In [50]:
if instances_to_terminate:
    response = ec2_client.terminate_instances(InstanceIds=instances_to_terminate)
    print(f"Terminated {len(instances_to_terminate)} instance(s):", instances_to_terminate)
else:
    print("No instances found with SSH open to the world")

Terminated 2 instance(s): ['i-018c484b8e50fe20c', 'i-0178ce093ceb0bedf']
